# Synthetic S-PLUS training set (part 1)

Bagpipes (BC03-MILES) models in the 12 S-PLUS bands, drawn by Latin Hypercube from wide priors fixed **before** looking at the data. All prior code lives in `synthetic_model.py`, shared with `spectroscopic_reference.ipynb`.

**Inputs:** `splus-filters/`, `dr3_corrected.csv` (S-PLUS DR3 corrected for Galactic extinction).
**Outputs:** `training_set.parquet` (noise-free), `synthetic_validation.parquet` and `synthetic_test.parquet` (S-PLUS noise and non-detections), `splus_noise_model.csv`, `color_color.png`, `coverage_criterion.png`.
**Run:** top to bottom with the *Inference4ETGs (MultiNest)* kernel, ~35 min on 11 workers.

**SFH:** 16 constant age bins up to the age of the Universe at $z$: ~50% Dirichlet(1) fractions; ~46% skew-normal (peak $\sim N(12, 3)$ Gyr ago, width $\sim N(1, 0.4)$, shape $U(-0.25, 0.75)$); ~4% skew-normal + burst (10–500 Myr ago, 40–50 Myr wide, 1–10% of the mass). Metallicity rises from $Z_{\rm ini} = 0.4 Z_{\rm fin}$ (oldest stars) to $Z_{\rm fin}$ with the mass formed, so the 16 `zmet_bin_*` labels carry a single free number, `Z_fin`.

| parameter | prior |
|---|---|
| $z$ | $U(0.002, 0.7)$ |
| $Z_{\rm fin}$ | log-uniform, $0.1$–$2.5\,Z_\odot$ |
| ISM $A_V$ | $U(0.01, 2)$ mag |
| extra birth-cloud $A_V$ (< 10 Myr) | $1.086\,\tau_{\rm BC}$, $\tau_{\rm BC}$ log-uniform 0.5–4 |
| $\log U$ | $U(-4, -1)$ |

**Labels:** `frac_bin_00…15` (mass fraction per bin, youngest first), `logsfr_bin_00…15` (log mean SFR per bin for $10^{10}\,M_\odot$ formed), `zmet_bin_00…15` ($Z/Z_\odot$), plus `log_mstar`, `log_sfr`, `mw_age`, `f_young`, `f_mid`, `f_old`. Bins older than the Universe have `frac = 0` and NaN SFR and metallicity. For an observed galaxy, $\Delta m = m_{\rm obs} - m_{\rm model}$ rescales every mass-like label by $-0.4\,\Delta m$ dex.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import qmc

import synthetic_model as sm

# Sizes and seeds
N_MODELS = 2**19                 # training set
N_HELD_OUT = 2**14               # each synthetic validation / test set
TRAINING_SEED = 0
VALIDATION_SEED = 1
TEST_SEED = 2
N_JOBS = 11
BLOCKS_PER_JOB = 20              # small blocks keep the workers busy until the end

# Files
SPLUS_FILE = "dr3_corrected.csv"
TRAINING_FILE = "training_set.parquet"
VALIDATION_FILE = "synthetic_validation.parquet"
TEST_FILE = "synthetic_test.parquet"
NOISE_MODEL_FILE = "splus_noise_model.csv"

MIN_PROB_GALAXY = 0.8
OLD_GEOMETRIC_COVERAGE = 0.991   # geometric test of commit c04268e (K = 2), kept for one comparison

## 1. Training set
Each model is a Latin Hypercube vector $u \in [0, 1)^{28}$; `sm.components` maps it to Bagpipes components by inverse CDFs, so the SFH shares and priors are exactly those of the table.

In [ ]:
def simulate_in_parallel(n_models, seed):
    """Latin Hypercube draw of n_models unit-cube vectors, simulated on N_JOBS workers."""
    sampler = qmc.LatinHypercube(len(sm.DIMS), seed=seed)
    unit_vectors = sampler.random(n_models)
    blocks = np.array_split(unit_vectors, BLOCKS_PER_JOB * N_JOBS)
    tables = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(sm.simulate_models)(block) for block in blocks)
    return pd.concat(tables, ignore_index=True)


training_models = simulate_in_parallel(N_MODELS, TRAINING_SEED)
training_models.to_parquet(TRAINING_FILE, index=False)

# Check: Bagpipes returned positive fluxes in every band for every model
finite_magnitudes = np.isfinite(training_models[sm.BANDS].to_numpy()).all(axis=1)
print(f"{len(training_models)} models saved to {TRAINING_FILE}")
assert finite_magnitudes.all(), f"{(~finite_magnitudes).sum()} models have non-finite magnitudes"
display(training_models[["redshift", "sfh_type", "Z_fin", "log_mstar", "log_sfr", "mw_age", "f_old"]].describe().T.round(3))

## 2. S-PLUS catalogue
A band is detected when its magnitude is measured and its error is below 1 mag. Galaxies need at least 6 detected bands and a photometric redshift.

In [ ]:
galaxy_catalogue = pd.read_csv(SPLUS_FILE)
galaxy_catalogue = galaxy_catalogue[galaxy_catalogue["PROB_GAL"] > MIN_PROB_GALAXY].reset_index(drop=True)

magnitude_columns = []
error_columns = []
for band in sm.BANDS:
    magnitude_columns.append(f"{band}_auto")
    error_columns.append(f"e_{band}_auto")

all_magnitudes = galaxy_catalogue[magnitude_columns].to_numpy()
all_errors = galaxy_catalogue[error_columns].to_numpy()
all_detected = sm.detected_bands(all_magnitudes, all_errors)

# Galaxies for the coverage test
enough_bands = all_detected.sum(axis=1) >= sm.MIN_DETECTED_BANDS
has_redshift = np.isfinite(galaxy_catalogue["zml"].to_numpy())
keep = enough_bands & has_redshift

splus = galaxy_catalogue[keep].reset_index(drop=True)
observed_magnitudes = all_magnitudes[keep]
observed_errors = all_errors[keep]
detected = all_detected[keep]
observed_z = splus["zml"].to_numpy()
observed_dz = sm.redshift_window(observed_z)

outside_prior = (observed_z < sm.Z_RANGE[0]) | (observed_z > sm.Z_RANGE[1])
print(f"{len(splus)} galaxies, {detected.sum(axis=1).mean():.1f} detected bands on average; "
      f"{outside_prior.sum()} with zml outside Z_RANGE")

## 3. Noise model
Per band and 0.25 mag bin of measured magnitude, the distribution of DR3 errors (21 quantiles of $\log_{10}$ error). A synthetic band is detected when its drawn error is below 1 mag, so faint bins reproduce DR3's non-detection rate.

In [ ]:
noise_table = sm.fit_noise_model(galaxy_catalogue)
noise_table.to_csv(NOISE_MODEL_FILE, index=False)
print(f"{len(noise_table)} (band, magnitude bin) rows saved to {NOISE_MODEL_FILE}")

# Median and 10-90% range of the errors in each band
figure, axes = plt.subplots(3, 4, figsize=(14, 9), sharex=True, sharey=True)
for axis, band in zip(axes.flat, sm.BANDS):
    band_rows = noise_table[noise_table["band"] == band]
    bin_centres = (band_rows["magnitude_low"] + band_rows["magnitude_high"]) / 2
    axis.fill_between(bin_centres, 10 ** band_rows["log_error_q02"], 10 ** band_rows["log_error_q18"],
                      color="grey", alpha=0.4, lw=0)
    axis.plot(bin_centres, 10 ** band_rows["log_error_q10"], color="black")
    axis.axhline(sm.MAX_DETECTED_ERROR, color="red", ls=":")
    axis.set(yscale="log", title=band)
for axis in axes[-1]:
    axis.set_xlabel("magnitude")
for axis in axes[:, 0]:
    axis.set_ylabel("error (mag)")
figure.tight_layout()
plt.show()

## 4. Synthetic validation and test sets
Independent draws with their own seeds. Each model takes the $r$ magnitude of a random DR3 galaxy at similar redshift, then gets S-PLUS errors, flux noise and non-detections. Mass-like labels are rescaled by the same magnitude offset.

In [ ]:
catalogue_r = galaxy_catalogue["r_auto"].to_numpy()
catalogue_z = galaxy_catalogue["zml"].to_numpy()
held_out_seeds = {VALIDATION_FILE: VALIDATION_SEED, TEST_FILE: TEST_SEED}

for output_file, seed in held_out_seeds.items():
    rng = np.random.default_rng(seed)
    models = simulate_in_parallel(N_HELD_OUT, seed)
    model_magnitudes = models[sm.BANDS].to_numpy()

    # 1. Realistic brightness at each redshift
    scaled_magnitudes, offsets = sm.scale_to_observed_r(model_magnitudes, models["redshift"].to_numpy(),
                                                        catalogue_r, catalogue_z, rng)
    for label in sm.MASS_DEPENDENT_LABELS:
        models[label] = models[label] - 0.4 * offsets
    models["magnitude_offset"] = offsets

    # 2. S-PLUS noise and non-detections
    noisy_magnitudes, errors, detected_mask = sm.add_splus_noise(scaled_magnitudes, noise_table, rng)
    for band_index, band in enumerate(sm.BANDS):
        models[band] = noisy_magnitudes[:, band_index]
        models[f"e_{band}"] = errors[:, band_index]
    models.to_parquet(output_file, index=False)

    print(f"{output_file}: {len(models)} models, {detected_mask.sum(axis=1).mean():.1f} detected bands "
          f"on average (DR3: {all_detected.sum(axis=1).mean():.1f})")

## 5. Coverage test v2
For each galaxy, only models with $|z_{\rm model} - z_{\rm ml}| < \Delta z$ are compared. Over the detected bands, $\chi^2$ is minimised analytically over the magnitude offset (the model's mass). The galaxy is covered if the best $\chi^2$ has $p > 0.01$ for $n_{\rm detected} - 1$ degrees of freedom; the 0.05 mag error floor stands in for model error. Noisy synthetic galaxies should be covered ~99% of the time; that is the test's calibration check.

In [ ]:
training_magnitudes = training_models[sm.BANDS].to_numpy()
training_z = training_models["redshift"].to_numpy()


def coverage_in_parallel(magnitudes, errors, detected_mask, z, dz):
    """sm.coverage_test on blocks of galaxies, on N_JOBS workers."""
    blocks = np.array_split(np.arange(len(z)), 4 * N_JOBS)
    tables = Parallel(n_jobs=N_JOBS)(
        delayed(sm.coverage_test)(magnitudes[block], errors[block], detected_mask[block], z[block], dz[block],
                                  training_magnitudes, training_z)
        for block in blocks
    )
    return pd.concat(tables, ignore_index=True)


# 1. S-PLUS galaxies
coverage = coverage_in_parallel(observed_magnitudes, observed_errors, detected, observed_z, observed_dz)
for column in coverage.columns:
    splus[column] = coverage[column].to_numpy()
print(f"Coverage v2 (p > {sm.P_MIN}): {splus['covered'].mean():.1%} of {len(splus)} galaxies")
print(f"Old geometric test (K = 2): {OLD_GEOMETRIC_COVERAGE:.1%}")

# 2. Calibration check on noisy synthetic galaxies
validation = pd.read_parquet(VALIDATION_FILE)
validation_magnitudes = validation[sm.BANDS].to_numpy()
validation_error_columns = []
for band in sm.BANDS:
    validation_error_columns.append(f"e_{band}")
validation_errors = validation[validation_error_columns].to_numpy()
validation_detected = sm.detected_bands(validation_magnitudes, validation_errors)
usable = validation_detected.sum(axis=1) >= sm.MIN_DETECTED_BANDS
validation_z = validation["redshift"].to_numpy()[usable]
validation_coverage = coverage_in_parallel(validation_magnitudes[usable], validation_errors[usable],
                                           validation_detected[usable], validation_z,
                                           sm.redshift_window(validation_z))
print(f"Synthetic validation galaxies covered: {validation_coverage['covered'].mean():.1%} "
      f"(expected about {1 - sm.P_MIN:.0%})")

# 3. Coverage by redshift, magnitude and number of detected bands
groupings = {
    "zml": pd.cut(splus["zml"], [-0.1, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 10]),
    "r_auto": pd.cut(splus["r_auto"], [0, 17, 18, 19, 20, 20.5, 21, 21.5, 30]),
    "n_detected": splus["n_dof"] + 1,
}
for name, grouping in groupings.items():
    grouped = splus.groupby(grouping, observed=True)
    table = pd.DataFrame({
        "covered": grouped["covered"].mean(),
        "median p": grouped["p_value"].median(),
        "galaxies": grouped.size(),
    })
    display(table.round(3))

## 6. Figures
Colour-colour diagrams of the synthetic domain with covered and uncovered galaxies, and the p-value distributions of S-PLUS and synthetic galaxies (uniform if models and errors are right).

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Poster palette
POSTER = {"panel": "#1b1b3a", "grid": "#2a2a52", "edge": "#3d3d70", "text": "#d6d6ec",
          "muted": "#8a8ab8", "cyan": "#5fd0f5", "red": "#e0524f", "red_edge": "#8c2424",
          "covered": "#b8b8d0"}
DOMAIN_COLOURMAP = LinearSegmentedColormap.from_list("poster_blue", ["#22305f", "#2f5fa0", "#4aa6e0", POSTER["cyan"]])
POSTER_STYLE = {"figure.facecolor": POSTER["panel"], "axes.facecolor": POSTER["panel"],
                "axes.edgecolor": POSTER["edge"], "axes.labelcolor": POSTER["text"], "axes.labelsize": 20,
                "xtick.color": POSTER["muted"], "ytick.color": POSTER["muted"], "text.color": POSTER["text"],
                "font.size": 16, "legend.frameon": False}
PANELS = [("g", "i", "r", "z"), ("u", "g", "g", "r"), ("g", "r", "r", "i"), ("r", "i", "i", "z")]
N_GALAXIES_SHOWN = 1500


def colour(magnitudes, first_band, second_band):
    """first_band - second_band from a (n, 12) magnitude array."""
    return magnitudes[:, sm.BANDS.index(first_band)] - magnitudes[:, sm.BANDS.index(second_band)]


covered = splus["covered"].to_numpy()
shown = np.random.default_rng(TRAINING_SEED).random(len(splus)) < N_GALAXIES_SHOWN / len(splus)

# Figure 1: colour-colour diagrams
with plt.rc_context(POSTER_STYLE):
    figure, axes = plt.subplots(2, 2, figsize=(12, 10))
    for axis, (band_a, band_b, band_c, band_d) in zip(axes.flat, PANELS):
        panel_detected = shown.copy()
        for band in (band_a, band_b, band_c, band_d):
            panel_detected = panel_detected & detected[:, sm.BANDS.index(band)]
        galaxy_x = colour(observed_magnitudes, band_a, band_b)[panel_detected]
        galaxy_y = colour(observed_magnitudes, band_c, band_d)[panel_detected]
        panel_covered = covered[panel_detected]

        x_limits = np.percentile(galaxy_x, [0.5, 99.5]) + [-0.8, 1.7]
        y_limits = np.percentile(galaxy_y, [0.5, 99.5]) + [-0.8, 0.95]
        axis.hexbin(colour(training_magnitudes, band_a, band_b), colour(training_magnitudes, band_c, band_d),
                    extent=(*x_limits, *y_limits), bins="log", mincnt=1, cmap=DOMAIN_COLOURMAP, zorder=1)
        axis.scatter(galaxy_x[panel_covered], galaxy_y[panel_covered], s=8, c=POSTER["covered"],
                     alpha=0.45, lw=0, zorder=2)
        axis.scatter(galaxy_x[~panel_covered], galaxy_y[~panel_covered], s=10, c=POSTER["red"],
                     edgecolors=POSTER["red_edge"], lw=0.5, zorder=3)
        axis.set(xlim=x_limits, ylim=y_limits, xlabel=f"${band_a} - {band_b}$", ylabel=f"${band_c} - {band_d}$")

    handles = [
        plt.Line2D([], [], ls="", marker="h", ms=10, color=POSTER["cyan"], label="Synthetic Domain"),
        plt.Line2D([], [], ls="", marker="o", ms=10, color=POSTER["covered"], label="Covered Galaxies"),
        plt.Line2D([], [], ls="", marker="o", ms=10, color=POSTER["red"], mec=POSTER["red_edge"], label="Uncovered Galaxies"),
    ]
    figure.legend(handles=handles, loc="upper center", ncol=3, labelcolor=POSTER["text"])
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    figure.savefig("color_color.png", dpi=300, facecolor=figure.get_facecolor())
    plt.show()

# Figure 2: p-value distributions
log_p_bins = np.linspace(-8, 0, 41)
with plt.rc_context(POSTER_STYLE):
    figure, axis = plt.subplots(figsize=(10, 7))
    splus_log_p = np.log10(np.clip(splus["p_value"].dropna(), 1e-8, 1))
    validation_log_p = np.log10(np.clip(validation_coverage["p_value"].dropna(), 1e-8, 1))
    axis.hist(splus_log_p, bins=log_p_bins, density=True, histtype="step", lw=2, color=POSTER["covered"], label="S-PLUS DR3")
    axis.hist(validation_log_p, bins=log_p_bins, density=True, histtype="step", lw=2, color=POSTER["cyan"], label="Synthetic validation")
    axis.axvline(np.log10(sm.P_MIN), color=POSTER["red"], ls="--", lw=2, label=f"$p = {sm.P_MIN}$")
    axis.set(xlabel=r"$\log_{10} p$", ylabel="density")
    axis.grid(True, color=POSTER["grid"])
    axis.legend(labelcolor=POSTER["text"])
    figure.tight_layout()
    figure.savefig("coverage_criterion.png", dpi=300, facecolor=figure.get_facecolor())
    plt.show()